In [29]:
# ===== Imports =====
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

import numpy as np
import matplotlib.pyplot as plt
import time
import copy
import os
import csv
import kagglehub
from tqdm import tqdm


from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [8]:
# Download Dataset
path = kagglehub.dataset_download("miljan/stanford-dogs-dataset-traintest")


print("Path to dataset files:", path)

100%|██████████| 393M/393M [01:07<00:00, 6.10MB/s] 

Extracting files...


Path to dataset files: C:\Users\melot\.cache\kagglehub\datasets\miljan\stanford-dogs-dataset-traintest\versions\1


In [9]:
def rename_classes(path):

    for name in os.listdir(path):

        old_path = os.path.join(path, name)

        if os.path.isdir(old_path):

            new_name = name.split("-",1)[1]

            new_path = os.path.join(path, new_name)

            os.rename(old_path, new_path)


rename_classes(os.path.join(path,"cropped/train"))
rename_classes(os.path.join(path,"cropped/test"))

In [18]:
print(os.path.join(path,"cropped/train"))

C:\Users\melot\.cache\kagglehub\datasets\miljan\stanford-dogs-dataset-traintest\versions\1\cropped/train


In [3]:
# ===== Transforms =====

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
# ===== Training Dataset =====
train_path = os.path.join(path, "cropped/train")

full_dataset = datasets.ImageFolder(train_path, transform=train_transforms)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# aplicar transform de teste na validação
val_dataset.dataset.transform = test_transforms

In [23]:
# ===== Test dataset =====
test_path = os.path.join(path, "cropped/test")

test_dataset = datasets.ImageFolder(test_path, transform=test_transforms)

In [56]:
# ===== DataLoaders =====
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=32, 
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
test_loader  = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True)

classes = full_dataset.classes
num_classes = len(classes)

print("Classes:", num_classes)

Classes: 120


In [62]:
# ===== Model =====
model = models.efficientnet_v2_s(pretrained=True)

# congelar backbone
for param in model.features.parameters():
    param.requires_grad = False

# substituir camada final
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

c:\Users\melot\anaconda3\envs\tf_gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\melot\anaconda3\envs\tf_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_S_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [51]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

Total params: 20,331,208
Trainable params: 153,720


In [63]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4  # L2 regularization
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=3,
    factor=0.3
)

In [64]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = np.inf
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

early_stopping = EarlyStopping(patience=5)

In [65]:
os.makedirs("models", exist_ok=True)

EPOCHS = 30

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_model = copy.deepcopy(model.state_dict())
best_loss = np.inf

start_time = time.time()

In [66]:
for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss, correct, total = 0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [TRAIN]")

    for batch_idx, (images, labels) in enumerate(pbar):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        #print("Batch:", images.device, labels.device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

        acc = correct / total

        # atualização em tempo real
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{acc:.4f}",
            "batch": f"{batch_idx+1}/{len(train_loader)}"
        })

    train_loss /= len(train_loader)
    train_acc = correct / total

    # ===== VALIDATION =====
    model.eval()
    val_loss, correct, total = 0, 0, 0

    vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [VAL]")

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(vbar):

            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

            acc = correct / total

            vbar.set_postfix({
                "val_loss": f"{loss.item():.4f}",
                "val_acc": f"{acc:.4f}"
            })

    val_loss /= len(val_loader)
    val_acc = correct / total

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"\nEpoch {epoch+1}: Train Acc={train_acc:.4f} | Val Acc={val_acc:.4f}\n")

    # salvar melhor modelo
    if val_loss < best_loss:
        best_loss = val_loss
        best_model = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "models/best_model.pth")

    # early stopping
    if early_stopping.step(val_loss):
        print("Early stopping!")
        break

Epoch 1/30 [VAL]: 100%|██████████| 75/75 [00:19<00:00,  3.75it/s, val_loss=1.0644, val_acc=0.7908]



Epoch 1: Train Acc=0.6196 | Val Acc=0.7908



Epoch 2/30 [VAL]: 100%|██████████| 75/75 [00:20<00:00,  3.61it/s, val_loss=0.8526, val_acc=0.7996]



Epoch 2: Train Acc=0.7788 | Val Acc=0.7996



Epoch 3/30 [TRAIN]:   9%|▊         | 26/300 [00:13<02:24,  1.90it/s, loss=0.6822, acc=0.8293, batch=26/300]


KeyboardInterrupt: 

In [67]:
start = time.time()

for i, (images, labels) in enumerate(train_loader):
    if i == 50:
        break

print("Tempo DataLoader:", time.time() - start)

Tempo DataLoader: 10.66421127319336


In [68]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

start = time.time()

for _ in range(50):
    outputs = model(images)

print("Tempo GPU:", time.time() - start)

Tempo GPU: 5.650851249694824
